In [ ]:
from docx import Document

file_path = "../data/raw/Animal_Registration_FS.docx"

document = Document(file_path)

print("Document loaded successfully!")
print("Number of paragraphs:", len(document.paragraphs))

In [ ]:
print("Paragraphs:", len(document.paragraphs))
print("Tables:", len(document.tables))

print("\n--- First 20 paragraphs ---")

for i, paragraph in enumerate(document.paragraphs[:20]):
    text = paragraph.text.strip()
    
    if text:
        print(f"{i}: [{paragraph.style.name}] {text}")

In [ ]:
print("--- Document headings ---")

for i, paragraph in enumerate(document.paragraphs):
    if paragraph.style.name.startswith("Heading"):
        print(f"{i}: [{paragraph.style.name}] {paragraph.text.strip()}")

In [ ]:
print(f"Number of tables: {len(document.tables)}")

for table_index, table in enumerate(document.tables):
    print(f"\n{'=' * 60}")
    print(f"TABLE {table_index + 1}")
    print(f"Rows: {len(table.rows)}, Columns: {len(table.columns)}")
    
    for row in table.rows[:5]:
        print(" | ".join(cell.text.strip().replace("\n", " ") for cell in row.cells))
    
    if len(table.rows) > 5:
        print("...")

In [ ]:
from docx.document import Document as _Document
from docx.table import Table
from docx.text.paragraph import Paragraph
from docx.oxml.text.paragraph import CT_P
from docx.oxml.table import CT_Tbl

In [ ]:
def iter_block_items(parent):
    """
    Yield paragraphs and tables in the order
    they appear in the Word document.
    """
    if isinstance(parent, _Document):
        parent_elm = parent.element.body
    else:
        parent_elm = parent._tc

    for child in parent_elm.iterchildren():
        if isinstance(child, CT_P):
            yield Paragraph(child, parent)
        elif isinstance(child, CT_Tbl):
            yield Table(child, parent)

In [ ]:
for index, block in enumerate(iter_block_items(document)):
    
    if isinstance(block, Paragraph):
        text = block.text.strip()
        
        if text:
            print(f"{index}: PARAGRAPH [{block.style.name}] {text}")
    
    elif isinstance(block, Table):
        print(
            f"{index}: TABLE "
            f"[{len(block.rows)} rows x {len(block.columns)} columns]"
        )

In [ ]:
from docx.table import Table
from docx.text.paragraph import Paragraph


def extract_blocks(document):
    blocks = []

    for block in iter_block_items(document):

        if isinstance(block, Paragraph):
            text = block.text.strip()

            if text:
                blocks.append({
                    "type": "paragraph",
                    "text": text,
                    "metadata": {
                        "style": block.style.name
                    }
                })

        elif isinstance(block, Table):
            rows = []

            for row in block.rows:
                cells = [
                    cell.text.strip().replace("\n", " ")
                    for cell in row.cells
                ]
                rows.append(cells)

            blocks.append({
                "type": "table",
                "text": rows,
                "metadata": {
                    "rows": len(rows),
                    "columns": len(block.columns)
                }
            })

    return blocks

In [ ]:
blocks = extract_blocks(document)

print("Total blocks:", len(blocks))

print("\nFirst 10 blocks:\n")

for i, block in enumerate(blocks[:10]):
    print(f"BLOCK {i}")
    print("Type:", block["type"])
    print("Metadata:", block["metadata"])
    print("Content:", block["text"])
    print("-" * 60)

In [ ]:
def add_section_context(blocks):
    current_section = None
    current_subsection = None

    for block in blocks:

        if block["type"] == "paragraph":
            style = block["metadata"]["style"]
            text = block["text"]

            if style == "Heading 1":
                current_section = text
                current_subsection = None

            elif style == "Heading 2":
                current_subsection = text

            block["metadata"]["section"] = current_section
            block["metadata"]["subsection"] = current_subsection

        else:
            # Tables inherit the current section context
            block["metadata"]["section"] = current_section
            block["metadata"]["subsection"] = current_subsection

    return blocks

In [ ]:
blocks = add_section_context(blocks)

In [ ]:
for i, block in enumerate(blocks):

    if block["metadata"].get("section") == "6. Animal Movement":
        print(f"BLOCK {i}")
        print("Type:", block["type"])
        print("Text:", block["text"])
        print("Section:", block["metadata"]["section"])
        print("Subsection:", block["metadata"]["subsection"])
        print("-" * 60)

In [ ]:
from dataclasses import dataclass, field
from typing import Any


@dataclass
class DocumentBlock:
    block_type: str
    content: Any
    metadata: dict = field(default_factory=dict)

In [ ]:
document_blocks = [
    DocumentBlock(
        block_type=block["type"],
        content=block["text"],
        metadata=block["metadata"]
    )
    for block in blocks
]

print("Document blocks:", len(document_blocks))
print(document_blocks[0])

In [ ]:
from dataclasses import dataclass, field
from typing import Any


@dataclass
class SpecificationDocument:
    blocks: list[DocumentBlock]
    metadata: dict[str, Any] = field(default_factory=dict)

    @property
    def block_count(self):
        return len(self.blocks)

In [ ]:
spec_document = SpecificationDocument(
    blocks=document_blocks
)

print("Total blocks:", spec_document.block_count)
print("Document metadata:", spec_document.metadata)

In [ ]:
def extract_document_metadata(blocks):
    metadata = {}

    for block in blocks:
        if block["type"] != "table":
            continue

        table = block["text"]

        if not table or len(table[0]) != 2:
            continue

        headers = [cell.strip().lower() for cell in table[0]]

        if headers == ["document item", "value"]:
            for row in table[1:]:
                if len(row) == 2:
                    key = row[0].strip()
                    value = row[1].strip()

                    if key and value:
                        metadata[key] = value

            break

    return metadata

In [ ]:
document_metadata = extract_document_metadata(blocks)

spec_document.metadata = document_metadata

print("Document metadata:")
for key, value in spec_document.metadata.items():
    print(f"{key}: {value}")

In [ ]:
@dataclass
class DocumentChunk:
    text: str
    metadata: dict[str, Any] = field(default_factory=dict)

In [ ]:
chunk = DocumentChunk(
    text="Example specification content",
    metadata={
        "section": "Example Section",
        "subsection": "Example Subsection"
    }
)

print(chunk)

In [ ]:
def create_chunks(blocks, document_metadata):
    chunks = []

    current_section = None
    current_subsection = None
    current_content = []

    def save_current_chunk():
        if not current_content:
            return

        text = "\n".join(current_content).strip()

        if text:
            chunks.append(
                DocumentChunk(
                    text=text,
                    metadata={
                        "document_id": document_metadata.get("Document ID"),
                        "document_version": document_metadata.get("Version"),
                        "source_filename": document_metadata.get("source_filename"),
                        "section": current_section,
                        "subsection": current_subsection
                    }
                )
            )

    for block in blocks:

        block_type = block["type"]
        text = block["text"]

        # Paragraph
        if block_type == "paragraph":

            style = block["metadata"]["style"]
            text = text.strip()

            if style == "Heading 1":
                save_current_chunk()

                current_section = text
                current_subsection = None
                current_content = []

            elif style == "Heading 2":
                save_current_chunk()

                current_subsection = text
                current_content = []

            else:
                if text:
                    current_content.append(text)

        # Table
        elif block_type == "table":

            table_lines = []

            for row in text:
                table_lines.append(" | ".join(row))

            table_text = "\n".join(table_lines)

            if table_text:
                current_content.append(table_text)

    # Save final chunk
    save_current_chunk()

    return chunks

In [ ]:
chunks = create_chunks(blocks, spec_document.metadata)
print("Total chunks:", len(chunks))

In [ ]:
print(chunks[12])

In [ ]:
print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks, start=1):
    print(
        f"Chunk {i}: "
        f"section={chunk.metadata.get('section')}, "
        f"subsection={chunk.metadata.get('subsection')}, "
        f"characters={len(chunk.text)}"
    )

In [ ]:
for i, chunk in enumerate(chunks):

    if chunk.metadata.get("section") == "5. Animal Birth Registration":

        print(f"\n{'=' * 70}")
        print(f"CHUNK {i + 1}")
        print("Section:", chunk.metadata["section"])
        print("Subsection:", chunk.metadata["subsection"])
        print("\nText:")
        print(chunk.text)

In [ ]:
for i, chunk in enumerate(chunks):

    print(
        f"Chunk {i + 1}: "
        f"{len(chunk.text)} characters | "
        f"Section: {chunk.metadata.get('section')} | "
        f"Subsection: {chunk.metadata.get('subsection')}"
    )

In [ ]:
from pathlib import Path

source_filename = Path(file_path).name

spec_document.metadata["source_filename"] = source_filename

print(spec_document.metadata)

In [ ]:
import importlib.util

print("sentence_transformers:",
      importlib.util.find_spec("sentence_transformers") is not None)

print("openai:",
      importlib.util.find_spec("openai") is not None)

In [ ]:
from sentence_transformers import SentenceTransformer

print("sentence-transformers is working")

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully")

In [ ]:
embedding_dimension = model.get_embedding_dimension()

print("Embedding dimension:", embedding_dimension)

In [ ]:
sample_text = chunks[12].text

sample_embedding = model.encode(sample_text)

print("Text:")
print(sample_text)

print("\nEmbedding type:", type(sample_embedding))
print("Embedding shape:", sample_embedding.shape)

In [ ]:
print("First 10 values of the embedding:")
print(sample_embedding[:10])

In [ ]:
chunk_texts = [chunk.text for chunk in chunks]

chunk_embeddings = model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("Number of embeddings:", len(chunk_embeddings))
print("Embedding shape:", chunk_embeddings.shape)

In [ ]:
for chunk, embedding in zip(chunks, chunk_embeddings):
    chunk.metadata["embedding"] = embedding

print("Embedding attached to all chunks.")

In [ ]:
print("Chunk 13 metadata:")
print(chunks[12].metadata.keys())

print("\nEmbedding shape:")
print(chunks[12].metadata["embedding"].shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
query = "Can an animal be moved after its death?"

query_embedding = model.encode([query])

print("Query embedding shape:", query_embedding.shape)

In [ ]:
similarities = cosine_similarity(
    query_embedding,
    chunk_embeddings
)[0]

print("Number of similarity scores:", len(similarities))

In [ ]:
ranked_results = sorted(
    enumerate(similarities),
    key=lambda x: x[1],
    reverse=True
)

for rank, (chunk_index, score) in enumerate(ranked_results[:5], start=1):
    chunk = chunks[chunk_index]

    print(f"\nRank {rank}")
    print(f"Score: {score:.4f}")
    print(f"Section: {chunk.metadata.get('section')}")
    print(f"Subsection: {chunk.metadata.get('subsection')}")
    print(f"Text: {chunk.text[:300]}...")

In [ ]:
def search_specification(query, top_k=5):
    """
    Search the specification using semantic similarity.
    """

    # Convert the query into an embedding
    query_embedding = model.encode([query])

    # Compare query embedding with all chunk embeddings
    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    # Rank chunks by similarity
    ranked_results = sorted(
        enumerate(similarities),
        key=lambda x: x[1],
        reverse=True
    )

    # Return the top matching chunks
    results = []

    for chunk_index, score in ranked_results[:top_k]:

        chunk = chunks[chunk_index]

        results.append({
            "score": float(score),
            "chunk": chunk
        })

    return results

In [ ]:
results = search_specification(
    "Can an animal be moved after its death?",
    top_k=5
)

for rank, result in enumerate(results, start=1):

    chunk = result["chunk"]

    print(f"\nRank {rank}")
    print(f"Score: {result['score']:.4f}")
    print(f"Section: {chunk.metadata.get('section')}")
    print(f"Subsection: {chunk.metadata.get('subsection')}")
    print(f"Text: {chunk.text[:300]}...")

In [ ]:
for rank, result in enumerate(results, start=1):

    chunk = result["chunk"]

    print(f"\n{'=' * 70}")
    print(f"RESULT {rank}")
    print(f"{'=' * 70}")

    print(f"Score      : {result['score']:.4f}")
    print(f"Document ID: {chunk.metadata.get('document_id')}")
    print(f"Version    : {chunk.metadata.get('document_version')}")
    print(f"Source     : {chunk.metadata.get('source_filename')}")
    print(f"Section    : {chunk.metadata.get('section')}")
    print(f"Subsection : {chunk.metadata.get('subsection')}")

    print("\nContent:")
    print(chunk.text)

In [ ]:
def build_context(results):
    """
    Build a context string from retrieved specification chunks.
    """

    context_parts = []

    for rank, result in enumerate(results, start=1):

        chunk = result["chunk"]
        score = result["score"]

        context_parts.append(
            f"""[Source {rank}]
Document ID: {chunk.metadata.get('document_id')}
Version: {chunk.metadata.get('document_version')}
Source: {chunk.metadata.get('source_filename')}
Section: {chunk.metadata.get('section')}
Subsection: {chunk.metadata.get('subsection')}
Similarity Score: {score:.4f}

Content:
{chunk.text}
"""
        )

    return "\n\n".join(context_parts)

In [ ]:
context = build_context(results)

print(context)

In [ ]:
### Step 16: Configure the LLM Client

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found. Check your .env file."
    )

client = OpenAI(api_key=api_key)

print("LLM client configured successfully.")

In [ ]:
### Step 17: Generate an Answer Using Retrieved Context

In [ ]:
query = "Can an animal be moved after its death?"

# Step 1: Retrieve relevant specification chunks
results = search_specification(query, top_k=5)

# Step 2: Build context from the retrieved chunks
context = build_context(results)

# Step 3: Ask the LLM to answer using the context
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": """
You are SpecGPT, an assistant that answers questions
about functional specification documents.

Use only the provided specification context to answer.
Do not invent requirements or business rules.
If the context does not contain enough information,
clearly say that the specification does not provide
enough information to answer.

Mention the relevant section or subsection when possible.
"""
        },
        {
            "role": "user",
            "content": f"""
Question:
{query}

Specification context:
{context}
"""
        }
    ],
    temperature=0
)

answer = response.choices[0].message.content

print("Question:", query)
print("\nAnswer:\n", answer)

In [ ]:
### Step 18: Display Retrieved Source Evidence

In [ ]:
print("QUESTION:")
print(query)

print("\nGENERATED ANSWER:")
print(answer)

print("\n" + "=" * 70)
print("RETRIEVED SOURCE EVIDENCE")
print("=" * 70)

for rank, result in enumerate(results, start=1):
    chunk = result["chunk"]

    print(f"\nSource {rank}")
    print("-" * 70)
    print(f"Similarity Score : {result['score']:.4f}")
    print(f"Document ID      : {chunk.metadata.get('document_id')}")
    print(f"Document Version : {chunk.metadata.get('document_version')}")
    print(f"Source File      : {chunk.metadata.get('source_filename')}")
    print(f"Section          : {chunk.metadata.get('section')}")
    print(f"Subsection       : {chunk.metadata.get('subsection')}")
    print("\nSupporting Content:")
    print(chunk.text)

In [ ]:
### Step 19: Test Different Specification Questions

In [ ]:
query = "What information is required to register an animal birth?"

results = search_specification(query, top_k=5)
context = build_context(results)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": """
You are SpecGPT, an assistant that answers questions
about functional specification documents.

Answer using only the provided specification context.
Do not invent requirements or business rules.
If the context does not contain enough information,
clearly say so.
Mention the relevant section or subsection when possible.
"""
        },
        {
            "role": "user",
            "content": f"""
Question:
{query}

Specification context:
{context}
"""
        }
    ],
    temperature=0
)

answer = response.choices[0].message.content

print("QUESTION:", query)
print("\nANSWER:\n", answer)

In [ ]:
print("RETRIEVED EVIDENCE FOR BIRTH REGISTRATION")
print("=" * 60)

for rank, result in enumerate(results, start=1):
    chunk = result["chunk"]

    print(f"\nSource {rank}")
    print(f"Section: {chunk.metadata.get('section')}")
    print(f"Subsection: {chunk.metadata.get('subsection')}")
    print(f"Similarity: {result['score']:.4f}")
    print("\nContent:")
    print(chunk.text)
    print("-" * 60)

In [ ]:
### Step 20: Test an Unanswerable Question

In [ ]:
query = "What is the annual maintenance cost of the system?"

results = search_specification(query, top_k=5)
context = build_context(results)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": """
You are SpecGPT, an assistant that answers questions
about functional specification documents.

Use only the provided specification context.
Do not invent facts, values, or business rules.
If the context does not contain enough information
to answer, clearly state that the specification
does not provide sufficient information.
Mention relevant sections when possible.
"""
        },
        {
            "role": "user",
            "content": f"""
Question:
{query}

Specification context:
{context}
"""
        }
    ],
    temperature=0
)

answer = response.choices[0].message.content

print("QUESTION:", query)
print("\nANSWER:\n", answer)

In [ ]:
### Step 21: Test the Refactored Document Loader

In [ ]:

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from specgpt.document_loader import load_document

file_path = PROJECT_ROOT / "data" / "raw" / "Animal_Registration_FS.docx"

loaded_blocks = load_document(file_path)

print("Blocks extracted:", len(loaded_blocks))
print("First block:", loaded_blocks[0])
print("Module test completed.")


In [ ]:
### Step 22: Test the Refactored Chunker

In [ ]:

from specgpt.chunker import (
    add_section_context,
    extract_document_metadata,
    create_chunks,
)

# Reuse the blocks extracted by the new document loader.
chunking_blocks = load_document(file_path)

# Extract metadata and prepare chunks.
document_metadata = extract_document_metadata(chunking_blocks)
document_metadata["source_filename"] = file_path.name

chunks_refactored = create_chunks(
    chunking_blocks,
    document_metadata,
)

print("Document metadata:")
print(document_metadata)

print("\nNumber of chunks:", len(chunks_refactored))

print("\nFirst chunk metadata:")
print(chunks_refactored[0].metadata)

print("\nFirst chunk text:")
print(chunks_refactored[0].text[:500])


In [ ]:
print("Original notebook chunks:", len(chunks))
print("Refactored chunks:", len(chunks_refactored))

print("\nOriginal chunk headings:")
for i, chunk in enumerate(chunks):
    print(i, chunk.metadata.get("section"), "|",
          chunk.metadata.get("subsection"))

print("\nRefactored chunk headings:")
for i, chunk in enumerate(chunks_refactored):
    print(i, chunk.metadata.get("section"), "|",
          chunk.metadata.get("subsection"))

In [ ]:
import inspect
print(inspect.getsource(create_chunks))

In [ ]:
import specgpt.chunker as chunker

print("Module function:", chunker.create_chunks)
print("Imported function:", create_chunks)
print("Same function:", chunker.create_chunks is create_chunks)

In [ ]:
import inspect
print(inspect.getsource(create_chunks))

In [ ]:
import specgpt.chunker as chunker

print("Module file:", chunker.__file__)
print("Available functions:")

for name in dir(chunker):
    if not name.startswith("_"):
        obj = getattr(chunker, name)
        if callable(obj):
            print(name, "->", getattr(obj, "__name__", type(obj).__name__))

In [ ]:
import specgpt.chunker as chunker

print("create_chunks name:", chunker.create_chunks.__name__)
print("Function line:", chunker.create_chunks.__code__.co_firstlineno)
print("Module:", chunker.create_chunks.__module__)

In [ ]:
import importlib
import specgpt.chunker as chunker

importlib.reload(chunker)

chunks_refactored = chunker.create_chunks(
    chunking_blocks,
    document_metadata,
)

print("Refactored chunk count:", len(chunks_refactored))

In [ ]:
for i, chunk in enumerate(chunks_refactored):
    print(
        i,
        "| Section:", chunk.metadata.get("section"),
        "| Subsection:", chunk.metadata.get("subsection")
    )

In [ ]:

import importlib
import specgpt.embeddings as embeddings_module

importlib.reload(embeddings_module)

embedding_model = embeddings_module.load_embedding_model()

refactored_embeddings = embeddings_module.embed_chunks(
    embedding_model,
    chunks_refactored,
)

print("Number of chunks:", len(chunks_refactored))
print("Embedding matrix shape:", refactored_embeddings.shape)
print("Embedding dimensions:", embedding_model.get_embedding_dimension())


In [ ]:

import importlib
import specgpt.retriever as retriever_module

importlib.reload(retriever_module)

test_query = "Can an animal be moved after its death?"

retrieved_results = retriever_module.retrieve_chunks(
    query=test_query,
    model=embedding_model,
    chunks=chunks_refactored,
    chunk_embeddings=refactored_embeddings,
    top_k=5,
)

print("Query:", test_query)
print("\nTop retrieved chunks:\n")

for rank, result in enumerate(retrieved_results, start=1):
    chunk = result["chunk"]

    print(f"Rank: {rank}")
    print(f"Similarity: {result['score']:.4f}")
    print(f"Section: {chunk.metadata.get('section')}")
    print(f"Subsection: {chunk.metadata.get('subsection')}")
    print(f"Content: {chunk.text[:250]}")
    print("-" * 60)


In [ ]:

import importlib
import specgpt.context_builder as context_builder_module

importlib.reload(context_builder_module)

retrieved_context = context_builder_module.build_context(
    retrieved_results
)

print(retrieved_context[:3000])


In [ ]:

import importlib
import specgpt.generator as generator_module

importlib.reload(generator_module)

test_query = "Can an animal be moved after its death?"

test_context = context_builder_module.build_context(
    retrieved_results
)

test_answer = generator_module.generate_answer(
    client=client,
    query=test_query,
    context=test_context,
)

print("QUESTION:", test_query)
print("\nANSWER:\n", test_answer)


In [ ]:
import importlib
import specgpt.pipeline as pipeline_module

importlib.reload(pipeline_module)

print("Pipeline module imported successfully.")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

file_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Animal_Registration_FS.docx"
)

prepared_document = pipeline_module.prepare_document(file_path)

print("Document preparation successful.")
print("Document metadata:", prepared_document["document_metadata"])
print("Number of chunks:", len(prepared_document["chunks"]))
print("Embedding matrix shape:", prepared_document["chunk_embeddings"].shape)

In [ ]:
test_query = "Can an animal be moved after its death?"

pipeline_result = pipeline_module.ask_question(
    query=test_query,
    prepared_document=prepared_document,
    client=client,
)

print("QUESTION:", pipeline_result["question"])
print("\nANSWER:\n", pipeline_result["answer"])

print("\nSUPPORTING SOURCES:")
for i, result in enumerate(pipeline_result["sources"], start=1):
    chunk = result["chunk"]
    print(
        f"{i}. Section: {chunk.metadata.get('section')} | "
        f"Subsection: {chunk.metadata.get('subsection')} | "
        f"Score: {result['score']:.4f}"
    )